<a href="https://colab.research.google.com/github/zty2020china/IOS13-SimulateTouch/blob/master/%E7%84%A6%E7%82%89%E7%BB%BC%E5%90%88%E4%BB%BF%E7%9C%9F%E5%B9%B3%E5%8F%B0V1_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""
焦炉专业封装软件 V1.5 - 数据库与热工仿真深度集成版 (Mac 网络修复版)
"""

import os
# ==========================================
# Mac 网络防拦截补丁 (解决 httpx.RemoteProtocolError)
# 强制屏蔽本地自检时的网络代理干扰
# ==========================================
os.environ["HTTP_PROXY"] = ""
os.environ["HTTPS_PROXY"] = ""
os.environ["NO_PROXY"] = "localhost,127.0.0.1,0.0.0.0"

import math
import gradio as gr
import pandas as pd
from pydantic import BaseModel

# ==========================================
# 领域数据模型 (Domain Models)
# ==========================================
class FuelGas(BaseModel):
    h2: float
    ch4: float
    co: float
    cmhn: float
    cmhn_c2h4_ratio: float
    h2s: float
    co2: float
    n2: float
    o2: float
    humidity_g_nm3: float
    temperature_c: float

class AirCondition(BaseModel):
    excess_ratio: float
    temperature_c: float
    relative_humidity: float

class CoalSample(BaseModel):
    moisture_mt: float
    coke_temp_c: float
    raw_gas_temp_c: float

class OvenSchedule(BaseModel):
    oven_count: int
    wet_coal_per_oven: float
    turnaround_time_h: float
    coking_rate: float

# ==========================================
# 模块 A：焦炉参数数据库引擎 (Database Engine)
# ==========================================
MOCK_DB = {
    "7.63 (内置参考)": {
        "炭化室长（mm）": "18560", "炭化室高（mm）": "7630", "炭化室平均宽（mm）": "450",
        "炭化室有效容积（m3）": "60.5", "每孔炭化室装煤量（t）": "57.3", "周转时间": "25.0",
        "成焦率": "75.0", "炭化室锥度（mm）": "50"
    },
    "JN43-804 (内置参考)": {
        "炭化室长（mm）": "14080", "炭化室高（mm）": "4300", "炭化室平均宽（mm）": "450",
        "炭化室有效容积（m3）": "23.8", "每孔炭化室装煤量（t）": "17.9", "周转时间": "18.0",
        "成焦率": "73.0", "炭化室锥度（mm）": "50"
    }
}

class OvenDatabase:
    def __init__(self, csv_path="焦炉常用参数 - 工作表1.csv"):
        self.db = MOCK_DB.copy()
        self.load_from_csv(csv_path)

    def load_from_csv(self, csv_path):
        if not os.path.exists(csv_path):
            print(f"⚠️ 未找到本地 CSV {csv_path}，已启用内置标准炉型库。")
            return

        df = None
        encodings = ['gbk', 'utf-8-sig', 'gb2312', 'gb18030']
        for enc in encodings:
            try:
                df = pd.read_csv(csv_path, encoding=enc, index_col=0)
                break
            except:
                continue

        if df is not None:
            try:
                df_t = df.T
                df_t.columns = [str(col).strip() for col in df_t.columns]
                for model_name, row_data in df_t.iterrows():
                    clean_data = {k: str(v) for k, v in row_data.items() if pd.notna(v)}
                    self.db[str(model_name).strip()] = clean_data
                print(f"✅ 成功从 CSV 加载 {len(df_t)} 种炉型！")
            except Exception as e:
                print(f"❌ 解析表结构失败: {e}")

    def get_models(self):
        return list(self.db.keys())

    def get_params(self, model_name):
        return self.db.get(model_name, {})

oven_db = OvenDatabase()

# ==========================================
# 模块 B：热工与燃烧计算核心 (Thermal Engine)
# ==========================================
class CokeOvenEngine:
    @staticmethod
    def calculate_saturation_vapor_pressure(t_c):
        return 6.112 * math.exp((17.62 * t_c) / (243.12 + t_c))

    @staticmethod
    def run_simulation(fuel: FuelGas, air: AirCondition, coal: CoalSample, schedule: OvenSchedule):
        total_comp = fuel.h2 + fuel.ch4 + fuel.co + fuel.cmhn + fuel.h2s + fuel.co2 + fuel.n2 + fuel.o2
        if abs(total_comp - 100.0) > 0.1:
            return {"error": f"燃料组分总和为 {total_comp:.1f}%，请检查使其等于 100%"}

        c2h4 = fuel.cmhn * fuel.cmhn_c2h4_ratio
        c6h6 = fuel.cmhn * (1.0 - fuel.cmhn_c2h4_ratio)

        lhv = (126.36 * fuel.co + 107.98 * fuel.h2 + 358.18 * fuel.ch4 +
               590.3 * c2h4 + 1400.0 * c6h6 + 234.0 * fuel.h2s)

        o2_demand = (0.5 * fuel.h2 + 0.5 * fuel.co + 2.0 * fuel.ch4 +
                     3.0 * c2h4 + 7.5 * c6h6 + 1.5 * fuel.h2s - fuel.o2) / 100.0
        v0_dry_air = o2_demand / 0.21

        p_sat = CokeOvenEngine.calculate_saturation_vapor_pressure(air.temperature_c)
        x_w = (air.relative_humidity / 100.0) * p_sat / 1013.25
        v_actual_wet_air = (air.excess_ratio * v0_dry_air) / (1 - x_w)
        v_h2o_from_air = v_actual_wet_air * x_w

        v_co2 = (fuel.co + fuel.co2 + fuel.ch4 + 2 * c2h4 + 6 * c6h6) / 100.0
        v_so2 = fuel.h2s / 100.0
        v_h2o_combust = (fuel.h2 + 2 * fuel.ch4 + 2 * c2h4 + 3 * c6h6 + fuel.h2s) / 100.0
        v_h2o_fuel_physical = fuel.humidity_g_nm3 * 0.00124
        v_n2_total = (fuel.n2 / 100.0) + 0.79 * air.excess_ratio * v0_dry_air
        v_o2_excess = 0.21 * (air.excess_ratio - 1) * v0_dry_air

        v_actual_wet_flue = v_co2 + v_so2 + v_n2_total + v_o2_excess + v_h2o_combust + v_h2o_fuel_physical + v_h2o_from_air

        dry_coal_per_oven = schedule.wet_coal_per_oven * (1 - coal.moisture_mt / 100.0)
        k_c = schedule.coking_rate / 100.0 if schedule.coking_rate > 1.0 else schedule.coking_rate

        annual_coke_tons = (schedule.oven_count / schedule.turnaround_time_h) * 8000 * schedule.wet_coal_per_oven * k_c
        hourly_dry_coal_kg = (schedule.oven_count / schedule.turnaround_time_h) * dry_coal_per_oven * 1000.0

        q2_water = (coal.moisture_mt / (100 - coal.moisture_mt)) * (2490 + 1.98 * coal.raw_gas_temp_c)
        c_coke = 1.05 + 0.0003 * coal.coke_temp_c
        q1_coke = k_c * c_coke * coal.coke_temp_c
        q_dry_estimated = (q1_coke + q2_water) / 0.60

        total_heat_demand_kj_h = hourly_dry_coal_kg * q_dry_estimated
        total_fuel_gas_nm3_h = total_heat_demand_kj_h / lhv
        total_waste_gas_nm3_h = total_fuel_gas_nm3_h * v_actual_wet_flue

        waste_gas_per_heating_wall = total_waste_gas_nm3_h / schedule.oven_count
        fuel_gas_per_heating_wall = total_fuel_gas_nm3_h / schedule.oven_count

        return {
            "capacity": annual_coke_tons,
            "metrics": {
                "【燃烧】燃料低位发热量 LHV": f"{lhv:.0f} kJ/Nm³",
                "【燃烧】理论干空气需量": f"{v0_dry_air:.3f} Nm³/Nm³",
                "【燃烧】实际湿废气生成率": f"{v_actual_wet_flue:.3f} Nm³/Nm³",
                "【热工】水分蒸发潜热支出": f"{q2_water:.0f} kJ/kg(干煤)",
                "【热工】综合干煤炼焦耗热量": f"{q_dry_estimated:.0f} kJ/kg(干煤)",
                "【宏观】全炉小时干煤处理量": f"{hourly_dry_coal_kg:.0f} kg/h",
                "【宏观】全炉总计加热煤气量": f"{total_fuel_gas_nm3_h:.0f} Nm³/h",
                "【分配】单室设计加热煤气量": f"{fuel_gas_per_heating_wall:.0f} Nm³/h",
                "【分配】单室设计废气排烟量": f"**{waste_gas_per_heating_wall:.0f} Nm³/h**"
            }
        }

# ==========================================
# 模块 C：Gradio 综合前端 (UI Integration)
# ==========================================
def auto_fill_fuel(gas_type):
    if gas_type == "焦炉煤气 (COG)": return 58.0, 25.0, 6.0, 2.5, 0.8, 0.5, 2.5, 5.0, 0.5, 10.0
    elif gas_type == "高炉煤气 (BFG)": return 2.5, 0.0, 24.0, 0.0, 1.0, 0.0, 17.5, 56.0, 0.0, 30.0
    else: return 15.0, 6.0, 20.0, 0.5, 0.8, 0.1, 14.0, 44.0, 0.4, 20.0

def load_oven_params(model_name):
    data = oven_db.get_params(model_name)
    charge = float(data.get("每孔炭化室装煤量（t）", 30.0))
    turnaround = float(data.get("周转时间", 20.0))
    coking_rate = float(data.get("成焦率", 75.0))

    dim_str = (f"**炉体物理轮廓**：全长 {data.get('炭化室长（mm）','-')} mm × "
               f"全高 {data.get('炭化室高（mm）','-')} mm × "
               f"均宽 {data.get('炭化室平均宽（mm）','-')} mm\n"
               f"**有效容积**：{data.get('炭化室有效容积（m3）','-')} m³ | **锥度**：{data.get('炭化室锥度（mm）','-')} mm")

    return charge, turnaround, coking_rate, dim_str

def execute_integrated_calc(fuel_type, h2, ch4, co, cmhn, c2h4_ratio, h2s, co2, n2, o2,
                            gas_humidity, gas_temp, alpha, air_temp, air_rh,
                            coal_mt, coke_temp, raw_gas_temp,
                            oven_count, wet_coal_per_oven, turnaround_time, coking_rate):

    fuel = FuelGas(h2=h2, ch4=ch4, co=co, cmhn=cmhn, cmhn_c2h4_ratio=c2h4_ratio,
                   h2s=h2s, co2=co2, n2=n2, o2=o2, humidity_g_nm3=gas_humidity, temperature_c=gas_temp)
    air = AirCondition(excess_ratio=alpha, temperature_c=air_temp, relative_humidity=air_rh)
    coal = CoalSample(moisture_mt=coal_mt, coke_temp_c=coke_temp, raw_gas_temp_c=raw_gas_temp)
    schedule = OvenSchedule(oven_count=oven_count, wet_coal_per_oven=wet_coal_per_oven,
                            turnaround_time_h=turnaround_time, coking_rate=coking_rate)

    res = CokeOvenEngine.run_simulation(fuel, air, coal, schedule)

    if "error" in res:
        return res["error"]

    capacity_val = res["capacity"]

    md_report = f"### 🎯 项目标定年产能：<span style='color:red'>{capacity_val/10000:.2f} 万吨/年</span> *(基于全年8000h有效作业时间)*\n\n"
    md_report += "| 核心工程指标 | 计算结果与评价 |\n| :--- | :--- |\n"
    for k, v in res["metrics"].items():
        md_report += f"| {k} | {v} |\n"

    return md_report

with gr.Blocks(title="焦炉专业封装软件 V1.5") as app:
    gr.Markdown("# 🏭 焦炉综合设计与热工仿真平台 (V1.5 深度集成版)")
    gr.Markdown("将炉型数据库、燃烧动力学、流体力学修正与全炉生产流转深度打通，一键输出单室截面设计依据。")

    with gr.Row():
        with gr.Column(scale=4):
            with gr.Accordion("📚 第一步：选择焦炉基础型号与规模", open=True):
                model_dd = gr.Dropdown(choices=oven_db.get_models(), value=oven_db.get_models()[0], label="加载炉型特征库")
                model_dims = gr.Markdown("加载中...")
                with gr.Row():
                    oven_cnt = gr.Number(value=65, label="焦炉孔数 N (孔)")
                    wet_coal = gr.Number(label="单孔装煤量 (t) [系统自动带入]")
                    turnaround = gr.Number(label="周转时间 (h) [系统自动带入]")
                    coke_rt = gr.Number(label="成焦率 (%) [系统自动带入]")

            with gr.Accordion("🔥 第二步：配置燃料燃烧与气象边界", open=False):
                fuel_tp = gr.Dropdown(choices=["焦炉煤气 (COG)", "高炉煤气 (BFG)", "混合煤气 (MG)"], value="焦炉煤气 (COG)", label="快速导入典型气源")
                with gr.Row():
                    h2 = gr.Number(value=58.0, label="H2 (%)"); ch4 = gr.Number(value=25.0, label="CH4 (%)"); co = gr.Number(value=6.0, label="CO (%)")
                with gr.Row():
                    cmhn = gr.Number(value=2.5, label="CmHn (%)"); c2h4_rt = gr.Slider(0, 1, value=0.8, label="C2H4在CmHn占比"); h2s = gr.Number(value=0.5, label="H2S (%)")
                with gr.Row():
                    co2 = gr.Number(value=2.5, label="CO2 (%)"); n2 = gr.Number(value=5.0, label="N2 (%)"); o2 = gr.Number(value=0.5, label="O2 (%)")
                with gr.Row():
                    gas_hum = gr.Number(value=10.0, label="煤气含湿 (g/Nm³)"); air_temp = gr.Slider(-20, 45, value=25, label="环境气温 (℃)"); air_rh = gr.Slider(0, 100, value=60, label="空气相对湿度 (%)")

            with gr.Accordion("⚙️ 第三步：配煤水热物性与调火制度", open=False):
                with gr.Row():
                    alpha = gr.Slider(1.0, 1.5, value=1.20, step=0.01, label="空气过剩系数 α")
                    coal_mt = gr.Slider(5.0, 15.0, value=10.0, step=0.1, label="入炉煤水分 Mt (%)")
                with gr.Row():
                    coke_tmp = gr.Number(value=1050, label="推焦中心温度 (℃)")
                    raw_gas_tmp = gr.Number(value=750, label="荒煤气导出温度 (℃)")

            calc_btn = gr.Button("🚀 执行全系统综合测算", variant="primary")

        with gr.Column(scale=5):
            gr.Markdown("### 📑 工程计算书输出 (自动生成)")
            report_panel = gr.Markdown(">> 请在左侧配置参数后点击执行。系统将根据炉型特征自动推演单室热流与废气分布...")

    model_dd.change(fn=load_oven_params, inputs=[model_dd], outputs=[wet_coal, turnaround, coke_rt, model_dims])
    fuel_tp.change(fn=auto_fill_fuel, inputs=[fuel_tp], outputs=[h2, ch4, co, cmhn, c2h4_rt, h2s, co2, n2, o2, gas_hum])
    app.load(fn=load_oven_params, inputs=[model_dd], outputs=[wet_coal, turnaround, coke_rt, model_dims])
    calc_btn.click(
        fn=execute_integrated_calc,
        inputs=[fuel_tp, h2, ch4, co, cmhn, c2h4_rt, h2s, co2, n2, o2,
                gas_hum, gas_hum, alpha, air_temp, air_rh,
                coal_mt, coke_tmp, raw_gas_tmp,
                oven_cnt, wet_coal, turnaround, coke_rt],
        outputs=[report_panel]
    )

if __name__ == "__main__":
    # 改为 127.0.0.1 彻底解决 Mac 自带防火墙/端口代理屏蔽 0.0.0.0 的问题
    app.launch(server_name="127.0.0.1", server_port=7860)

⚠️ 未找到本地 CSV 焦炉常用参数 - 工作表1.csv，已启用内置标准炉型库。
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6a58f70af810ca5f41.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
